# Customer Segmentation & Transparent Churn-Risk Model

**Goal:** Segment customers by behavior/account characteristics and create a simple, reproducible, rule-based churn-risk score.

### Passing requirements
1. Choose useful behavioral and account features.
2. Create and explain customer segments/cohorts.
3. Design a reproducible churn-risk score and validate edge cases.
4. Present the work and a short explanation in one shareable notebook.

> 


## 1. Import libraries and load the dataset

We will use:
- **Pandas** for data handling
- **Matplotlib** for simple visualizations
- **NumPy** for numerical checks

The uploaded CSV is used directly in this notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

file_path = "07b2f134-bf2b-475f-bcaf-f2186ef1aecf.csv"
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
df.head()


## 2. Quick data-quality check

Before building segments or a risk score, check:
- column names and data types
- missing values
- duplicate customer IDs
- valid churn labels

These checks make the analysis reproducible and help avoid scoring bad records.


In [ ]:
print("Data types:")
display(df.dtypes)

print("\nMissing values:")
display(df.isna().sum())

print("\nDuplicate CustomerIDs:", df["CustomerID"].duplicated().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nChurn values:")
display(df["Churn"].value_counts())


### Data-quality decision

For this dataset:
- `CustomerID` is an identifier, so it is **not used as a predictive feature**.
- `Gender` and `Age` are also excluded from the churn-risk score because they are not necessary for this transparent behavioral/account rule set.
- The selected variables focus on **tenure, contract commitment, support activity, price, and payment method**.
- No missing-value treatment is required if the checks above show zero missing values.


## 3. Choose behavioral and account features

The risk score uses five easy-to-explain rules:

| Feature | Rule | Points |
|---|---|---:|
| Contract type | Month-to-Month | 30 |
| Tenure | ≤ 6 months | 25 |
| Support tickets | ≥ 4 tickets | 25 |
| Monthly charges | ≥ ₹80 | 10 |
| Payment method | UPI | 10 |

**Maximum score = 100.**

The weights are intentionally simple so that a business user can understand exactly why a customer received a score.


## 4. Create customer cohorts/segments

We will create tenure-based customer cohorts:

- **New Customer:** 0–6 months
- **Developing Customer:** 7–24 months
- **Loyal Customer:** more than 24 months

This is a cohort/segment based on account maturity rather than the target `Churn` column, so it can be used for customer targeting.


In [ ]:
def assign_segment(tenure):
    if tenure <= 6:
        return "New Customer"
    elif tenure <= 24:
        return "Developing Customer"
    else:
        return "Loyal Customer"

df["CustomerSegment"] = df["TenureMonths"].apply(assign_segment)

segment_summary = (
    df.groupby("CustomerSegment")
      .agg(
          Customers=("CustomerID", "count"),
          AvgTenureMonths=("TenureMonths", "mean"),
          AvgMonthlyCharges=("MonthlyCharges", "mean"),
          AvgSupportTickets=("SupportTickets", "mean"),
          ChurnRate=("Churn", lambda x: (x == "Yes").mean() * 100)
      )
      .round(2)
      .sort_index()
)

segment_summary


In [ ]:
plt.figure(figsize=(8, 5))
segment_counts = df["CustomerSegment"].value_counts()
plt.bar(segment_counts.index, segment_counts.values)
plt.title("Customers by Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Number of Customers")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


### Segment interpretation

The segments help answer a simple business question: **where are customers in their lifecycle?**

- New customers may need onboarding and early engagement.
- Developing customers may benefit from retention offers and support follow-up.
- Loyal customers can be targeted with loyalty benefits and renewal incentives.

The churn rate is shown as a descriptive validation metric, not as an input into the segment definition.


## 5. Build the transparent churn-risk score

Each rule contributes points when its condition is true.

This is deliberately **rule-based rather than a black-box machine-learning model**. A customer can therefore be given a clear explanation such as:

> "High risk because the customer is on a month-to-month contract, has high support activity, and has short tenure."

The score is calculated without using the actual `Churn` value.


In [ ]:
# Individual rule flags
df["Rule_MonthToMonth"] = (df["ContractType"] == "Month-to-Month").astype(int)
df["Rule_ShortTenure"] = (df["TenureMonths"] <= 6).astype(int)
df["Rule_HighSupport"] = (df["SupportTickets"] >= 4).astype(int)
df["Rule_HighMonthlyCharge"] = (df["MonthlyCharges"] >= 80).astype(int)
df["Rule_UPI"] = (df["PaymentMethod"] == "UPI").astype(int)

# Weighted score
df["ChurnRiskScore"] = (
    df["Rule_MonthToMonth"] * 30
    + df["Rule_ShortTenure"] * 25
    + df["Rule_HighSupport"] * 25
    + df["Rule_HighMonthlyCharge"] * 10
    + df["Rule_UPI"] * 10
)

def risk_band(score):
    if score >= 60:
        return "High Risk"
    elif score >= 30:
        return "Medium Risk"
    else:
        return "Low Risk"

df["RiskBand"] = df["ChurnRiskScore"].apply(risk_band)

df[[
    "CustomerID", "CustomerSegment", "ChurnRiskScore", "RiskBand",
    "ContractType", "TenureMonths", "SupportTickets",
    "MonthlyCharges", "PaymentMethod", "Churn"
]]


## 6. Generate an explanation for every score

A transparent score should not only produce a number. It should also explain the rules that triggered.


In [ ]:
def explain_risk(row):
    reasons = []

    if row["Rule_MonthToMonth"]:
        reasons.append("Month-to-month contract (+30)")
    if row["Rule_ShortTenure"]:
        reasons.append("Short tenure ≤ 6 months (+25)")
    if row["Rule_HighSupport"]:
        reasons.append("High support activity ≥ 4 tickets (+25)")
    if row["Rule_HighMonthlyCharge"]:
        reasons.append("Monthly charges ≥ ₹80 (+10)")
    if row["Rule_UPI"]:
        reasons.append("UPI payment method (+10)")

    return "; ".join(reasons) if reasons else "No risk rules triggered"

df["RiskExplanation"] = df.apply(explain_risk, axis=1)

display(
    df[["CustomerID", "ChurnRiskScore", "RiskBand", "RiskExplanation"]]
    .sort_values("ChurnRiskScore", ascending=False)
)


## 7. Validate the risk score against observed churn

For this learning exercise, we can compare the rule-based risk bands with the historical `Churn` column.

A useful first check is the churn rate within each risk band.


In [ ]:
risk_summary = (
    df.groupby("RiskBand", observed=False)
      .agg(
          Customers=("CustomerID", "count"),
          AvgRiskScore=("ChurnRiskScore", "mean"),
          ActualChurnRate=("Churn", lambda x: (x == "Yes").mean() * 100)
      )
      .round(2)
)

risk_summary


In [ ]:
# Cross-tabulation of predicted risk band vs actual churn
risk_churn_table = pd.crosstab(
    df["RiskBand"],
    df["Churn"],
    margins=True
)

risk_churn_table


### Interpretation

The validation table checks whether higher rule-based risk generally corresponds to more observed churn.

Because this dataset contains only 15 customers, the result should **not** be treated as statistically reliable or production-ready. It is evidence that the rules behave sensibly on this sample.


## 8. Edge-case validation

A reproducible scoring system should behave correctly at the boundaries.

We test:
1. A completely low-risk customer → expected score **0**
2. A customer triggering every rule → expected score **100**
3. Boundary values such as tenure = 6, support tickets = 4, and monthly charges = ₹80
4. Risk-band boundaries: 29 = Low, 30 = Medium, 59 = Medium, 60 = High


In [ ]:
# Test helper for the score
def calculate_risk_score(contract, tenure, tickets, monthly_charge, payment):
    return (
        (contract == "Month-to-Month") * 30
        + (tenure <= 6) * 25
        + (tickets >= 4) * 25
        + (monthly_charge >= 80) * 10
        + (payment == "UPI") * 10
    )

edge_cases = pd.DataFrame([
    {
        "Case": "All rules false",
        "Contract": "Two Year", "Tenure": 24, "Tickets": 0,
        "MonthlyCharge": 49.99, "Payment": "Credit Card",
        "ExpectedScore": 0
    },
    {
        "Case": "All rules true",
        "Contract": "Month-to-Month", "Tenure": 6, "Tickets": 4,
        "MonthlyCharge": 80, "Payment": "UPI",
        "ExpectedScore": 100
    },
    {
        "Case": "Boundary values",
        "Contract": "One Year", "Tenure": 6, "Tickets": 4,
        "MonthlyCharge": 80, "Payment": "Credit Card",
        "ExpectedScore": 60
    }
])

edge_cases["CalculatedScore"] = edge_cases.apply(
    lambda r: calculate_risk_score(
        r["Contract"], r["Tenure"], r["Tickets"],
        r["MonthlyCharge"], r["Payment"]
    ),
    axis=1
)

edge_cases["Pass"] = edge_cases["CalculatedScore"] == edge_cases["ExpectedScore"]
edge_cases


In [ ]:
# Test risk-band boundaries
boundary_tests = pd.DataFrame({
    "Score": [0, 29, 30, 59, 60, 100]
})

boundary_tests["RiskBand"] = boundary_tests["Score"].apply(risk_band)
boundary_tests


In [ ]:
# Automated assertions
assert df["ChurnRiskScore"].between(0, 100).all()
assert edge_cases["Pass"].all()
assert risk_band(29) == "Low Risk"
assert risk_band(30) == "Medium Risk"
assert risk_band(59) == "Medium Risk"
assert risk_band(60) == "High Risk"

print("All validation checks passed.")


## 9. Prioritize customers for retention

The highest-risk customers are the first candidates for retention outreach.

Suggested actions:
- **High Risk:** proactive retention call, service recovery, contract incentive
- **Medium Risk:** targeted engagement and support follow-up
- **Low Risk:** loyalty/upsell communication rather than urgent retention action


In [ ]:
priority_list = (
    df[[
        "CustomerID", "CustomerSegment", "ChurnRiskScore",
        "RiskBand", "RiskExplanation"
    ]]
    .sort_values(["ChurnRiskScore", "CustomerID"], ascending=[False, True])
)

priority_list


## 10. Final business insights

### Key findings from this sample

- The risk score is **transparent and reproducible**: every point comes from an explicit rule.
- Short tenure, month-to-month contracts, and high support activity are treated as the strongest warning signals in this demonstration.
- Customer lifecycle segmentation separates **New, Developing, and Loyal** customers.
- The risk explanation makes the model easier for a business stakeholder to audit and act on.
- Edge-case tests confirm that the score stays within **0–100** and that the defined thresholds behave consistently.

### Limitations

This dataset has only **15 customers**, so the rules and validation results are illustrative. For a real business model, the thresholds and weights should be selected using a larger historical dataset, business input, and proper out-of-sample validation.


## 11. One-paragraph submission explanation

**Customer Segmentation & Recommendation Model:** I segmented customers into New, Developing, and Loyal cohorts using tenure, then created a transparent 0–100 churn-risk score based on contract type, tenure, support-ticket activity, monthly charges, and payment method. Each rule has a documented weight, and every customer receives an explanation showing which rules contributed to the score. I validated the model using churn-rate comparisons and cross-tabulation, then tested boundary and edge cases to confirm reproducibility. The resulting priority list can be used to focus retention actions on high-risk customers first. Because the sample is small, the model is intended as a transparent demonstration rather than a production churn predictor.
